# M2.2 — Core model development: iterating on the fraud detector

**Track M: AI Capstone — M2.2 checkpoint.** This notebook continues the J1 fraud work
([01_supervised_fraud.ipynb](01_supervised_fraud.ipynb)). The code it uses lives in the tested `fraud/` package
(`fraud/data.py`, `features.py`, `metrics.py`, `model.py`, `experiments.py`); the notebook holds the experiments and the
reasoning.

**Baseline (M2.1).** The model committed at the end of J1 (`d3eae82`): XGBoost, engineered features, hyper-parameters
from the J1 search (rank 1), decision threshold = max F1 on the validation set. On the test set it scored PR-AUC 0.9972
with 4 missed frauds and 0 false alarms (`reports/model_comparison.csv`).

**The difficulty.** That baseline is close to the ceiling, so a large headline gain is not available and a small one can
be noise. The rules below are therefore fixed *before* any iteration is run.

## Rules written before the iterations

1. **Noise floor.** Refit the baseline with 5 different XGBoost seeds and take the standard deviation of validation
   PR-AUC; call it σ. A difference smaller than σ is not evidence of anything.
2. **Iterations are judged on the validation set only.** The test set is used twice in total: once for the baseline
   (§2) and once for the final model (notebook 05), as in J1.
3. **Keep rule.** Keep an iteration if validation PR-AUC rises by more than 2σ, **or** if expected cost (defined in §3,
   judged at a review cost of 100 units per alert) falls while PR-AUC does not drop by more than σ. Otherwise revert it.
   A reverted iteration stays in the log. *(Clarification made when the cost metric was written, before any iteration
   was run: the cost is defined in §3, not in iteration 2.)*
4. **Final claim.** Paired Poisson bootstrap of the test-set PR-AUC difference, final − baseline, 1,000 resamples. If
   the 95% interval excludes 0 it is a measurable improvement; if it includes 0 the result is reported as "no
   measurable gain" with the reason (ceiling / noise).

| § | Section |
|---|---|
| 1 | Setup, data and split |
| 2 | Iteration 0: reproduce the baseline and measure the noise floor |
| 3 | The cost metric, and the baseline's cost |
| 4 | Iteration 1: ratio and flag features |
| 5 | Iteration 2: a cost-based threshold |
| 6 | Iteration 3: a second-stage search on the full training set |
| 7 | What carries over (the test-set comparison is in notebook 05) |

## 1. Setup, data and split

In [1]:
import sys
import time
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold

# Run from notebooks/ (nbconvert) or from the project root; either way `import fraud` has to work.
ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT))

from fraud.data import SEED, load_paysim, scope_transfer_cashout, split_70_15_15
from fraud.experiments import log_iteration
from fraud.features import RATIO_COLS, add_ratio_features, make_features
from fraud.metrics import best_cost_threshold, best_f1_threshold, evaluate
from fraud.model import baseline_params, fit_xgb

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [2]:
d = scope_transfer_cashout(load_paysim())
X, RAW, ENG = make_features(d)
y = d["isFraud"].to_numpy()
train_idx, val_idx, test_idx = split_70_15_15(y)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

split_tbl = pd.DataFrame({"rows": [len(train_idx), len(val_idx), len(test_idx)],
                          "frauds": [y_train.sum(), y_val.sum(), y_test.sum()]},
                         index=["train", "validation", "test"])
split_tbl["fraud_rate"] = split_tbl["frauds"] / split_tbl["rows"]
display(split_tbl.style.format({"rows": "{:,}", "frauds": "{:,}", "fraud_rate": "{:.3%}"}))

# The same split as J1 (notebook 01, section 3): the test set must be the same 415,562 rows with 1,232 frauds.
assert len(test_idx) == 415_562 and y_test.sum() == 1_232

,rows,frauds,fraud_rate
train,"1,939,286","5,749",0.296%
validation,"415,561","1,232",0.296%
test,"415,562","1,232",0.296%


The split is the J1 split (same seed, same function logic), so the baseline can be compared with the committed numbers
row for row.

## 2. Iteration 0: reproduce the baseline and measure the noise floor

### 2.1 Reproduce the baseline
`run()` is the one place where a model is fitted and scored. It fits XGBoost with early stopping on the validation set,
scores the validation rows, and picks the max-F1 threshold on them. Every later iteration calls it with different
columns or parameters, so the comparison between iterations changes one thing at a time.

In [3]:
PARAMS = baseline_params()
print("baseline hyper-parameters (rank 1 of the J1 search):", PARAMS)


def run(cols, params, seed=SEED, frame=None):
    """Fit on train with early stopping on validation; return the model, its validation scores and its threshold."""
    frame = X if frame is None else frame
    model, secs = fit_xgb(frame.iloc[train_idx][cols], y_train, frame.iloc[val_idx][cols], y_val, params, seed)
    val_score = model.predict_proba(frame.iloc[val_idx][cols])[:, 1]
    return {"model": model, "secs": secs, "trees": model.best_iteration + 1, "val_score": val_score,
            "thr": best_f1_threshold(y_val, val_score)}


base = run(ENG, PARAMS)
base_val = evaluate("XGBoost baseline", y_val, base["val_score"], base["thr"], "engineered", base["secs"])
base_test = evaluate("XGBoost baseline", y_test, base["model"].predict_proba(X.iloc[test_idx][ENG])[:, 1],
                     base["thr"], "engineered", base["secs"])
print(f"{base['trees']} trees after early stopping, fitted in {base['secs']:.0f}s, threshold {base['thr']:.3f}")

baseline hyper-parameters (rank 1 of the J1 search): {'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.6, 'min_child_weight': 5, 'scale_pos_weight': 18.3}


117 trees after early stopping, fitted in 43s, threshold 0.506


In [4]:
committed = pd.read_csv(ROOT / "reports" / "model_comparison.csv").set_index("model").loc["XGBoost"]
check = pd.DataFrame({"J1 notebook (committed)": [committed["PR-AUC"], committed["FN"], committed["FP"]],
                      "this notebook (test set)": [base_test["PR-AUC"], base_test["FN"], base_test["FP"]]},
                     index=["PR-AUC", "missed frauds (FN)", "false alarms (FP)"])
display(check)
assert abs(base_test["PR-AUC"] - committed["PR-AUC"]) < 5e-4, "baseline no longer matches the committed J1 result"

,J1 notebook (committed),this notebook (test set)
PR-AUC,0.9972,0.9972
missed frauds (FN),4.0000,4.0000
false alarms (FP),0.0000,0.0000


### 2.2 Noise floor (rule 1)
Same data, same hyper-parameters, five XGBoost seeds (row/column subsampling is random, so the seed changes the model).
The spread of validation PR-AUC across seeds is what "no change" looks like.

In [5]:
rows = []
for s in [SEED, 1, 2, 3, 4]:
    r = run(ENG, PARAMS, seed=s)
    rows.append({"seed": s, "trees": r["trees"], "val PR-AUC": average_precision_score(y_val, r["val_score"]),
                 "fit seconds": r["secs"]})
noise = pd.DataFrame(rows)
sigma = float(noise["val PR-AUC"].std(ddof=1))
display(noise.style.format({"val PR-AUC": "{:.5f}", "fit seconds": "{:.0f}"}).hide(axis="index"))
display(Markdown(f"**σ = {sigma:.5f}** (sample standard deviation of validation PR-AUC over 5 seeds). "
                 f"Keep threshold for an iteration, rule 3: gain > 2σ = **{2 * sigma:.5f}**."))

seed,trees,val PR-AUC,fit seconds
42,117,0.99718,43
1,111,0.99724,41
2,22,0.99674,18
3,104,0.99737,39
4,111,0.99738,40


**σ = 0.00026** (sample standard deviation of validation PR-AUC over 5 seeds). Keep threshold for an iteration, rule 3: gain > 2σ = **0.00052**.

## 3. The cost metric, and the baseline's cost

F1 treats a missed fraud and a false alarm as equally bad. They are not: a missed fraud costs the money in it, a false
alarm costs an analyst's time. J1 already listed this as a limitation (notebook 01, §7.3). So every operating point is
also scored with

> **expected cost = (amount of the frauds that were not flagged) + (review cost × number of alerts raised)**

The review cost per alert is **an assumption**, not something PaySim tells us. PaySim amounts run from tens to millions
of units, so 10, 100 and 1,000 units per alert span a cheap to an expensive review. Rule 3 is judged at **100**; the other
two are a sensitivity check. Lower is better.

The baseline's operating point is the max-F1 threshold from J1.

In [6]:
from fraud.metrics import PRIMARY_REVIEW_COST, REVIEW_COSTS, expected_cost, missed_amount

amount_val = d["amount"].to_numpy()[val_idx]   # money in each validation transaction, for the cost metric


def cost_table(score, thr):
    """Validation operating point at `thr`: alerts, missed money, and expected cost at each review cost."""
    flagged = score >= thr
    row = {"alerts": int(flagged.sum()), "missed frauds": int(((y_val == 1) & ~flagged).sum()),
           "missed amount": missed_amount(y_val, amount_val, flagged)}
    for c in REVIEW_COSTS:
        row[f"cost @ {c}"] = expected_cost(y_val, amount_val, flagged, c)
    return row


display(pd.DataFrame([cost_table(base["val_score"], base["thr"])], index=["baseline, max-F1 threshold"])
        .style.format("{:,.0f}"))

,alerts,missed frauds,missed amount,cost @ 10,cost @ 100,cost @ 1000
"baseline, max-F1 threshold","1,227",5,"2,308,170","2,320,440","2,430,870","3,535,170"


At review costs of 10 and 100 nearly all of the baseline's cost (about 99% and 95%) is the money in the 5 frauds it
misses, not the cost of its 1,227 alerts. Only at 1,000 per alert do the alerts become a third of the cost. So the
F1-optimal operating point leaves money on the table unless alerts are very expensive, which is what iteration 2 tests.

In [7]:
COST_KEY = f"val_cost_c{PRIMARY_REVIEW_COST}"


def summarise(r, thr=None):
    """The validation numbers that go in the experiment log, for one fitted run (at its own threshold unless given)."""
    thr = r["thr"] if thr is None else thr
    cost = cost_table(r["val_score"], thr)
    return {"val_PR-AUC": average_precision_score(y_val, r["val_score"]),
            "val_FN": cost["missed frauds"], "val_FP": cost["alerts"] - (int(y_val.sum()) - cost["missed frauds"]),
            "val_missed_amount": cost["missed amount"], "val_alerts": cost["alerts"],
            COST_KEY: cost[f"cost @ {PRIMARY_REVIEW_COST}"], "threshold": thr, "trees": r["trees"],
            "fit_seconds": round(r["secs"], 1)}


def decide(row, ref):
    """Rule 3. Keep if PR-AUC rose by more than 2 sigma, or the cost fell without PR-AUC dropping by more than sigma."""
    gain = row["val_PR-AUC"] - ref["val_PR-AUC"]
    by_prauc = gain > 2 * sigma
    by_cost = row[COST_KEY] < ref[COST_KEY] and gain > -sigma
    return ("keep" if by_prauc or by_cost else "revert"), gain, by_prauc, by_cost


base_row = summarise(base)
assert base_row["val_FN"] == base_val["FN"] and base_row["val_FP"] == base_val["FP"]   # the two ways of counting agree
log_iteration({
    "iteration": 0, "name": "baseline (M2.1)",
    "hypothesis": "the refactored package reproduces the J1 model",
    "change": "none: J1 tuned XGBoost, engineered features, max-F1 threshold",
    **base_row, "sigma_seed": sigma, "decision": "reference",
})
pd.read_csv(ROOT / "reports" / "m2_2" / "experiment_log.csv")

,iteration,name,hypothesis,change,val_PR-AUC,val_FN,val_FP,trees,fit_seconds,sigma_seed,decision,val_missed_amount,val_alerts,val_cost_c100,threshold
0,0,baseline (M2.1),the refactored package reproduces the J1 model,"none: J1 tuned XGBoost, engineered features, m...",0.9972,5,0,117,42.5000,0.0003,reference,"2,308,170.0900","1,227.0000","2,430,870.0900",0.5057
1,1,ratio and flag features,trees cannot form ratios of two columns; maybe...,"add amountToOldBalanceOrig, amountEqualsOldBal...",0.9973,5,1,58,21.3000,0.0003,revert,"2,308,170.0900","1,228.0000","2,430,970.0900",0.1404
2,2,cost-based threshold,missed frauds carry most of the cost and score...,choose the threshold on validation by minimum ...,0.9972,3,589,117,18.9000,0.0003,keep,"1,597,908.4700","1,818.0000","1,779,708.4700",0.0406
3,3,"second-stage search, full training set",J1 searched 20 configurations on 30% of the tr...,"12-config grid around the J1 winner, scored by...",0.9971,3,1145,78,33.8000,0.0003,revert,"1,597,908.4700","2,374.0000","1,835,308.4700",0.0300


## 4. Iteration 1 — ratio and flag features

### 4.1 What are the baseline's validation errors?
Before adding anything, look at what the baseline gets wrong **on the validation set** (the test set is not looked at),
and at the **training set** to see whether those errors are a pattern or an accident.

In [8]:
V = X.iloc[val_idx][ENG].copy()
V["type"] = d["type"].iloc[val_idx].to_numpy()
V["isFraud"] = y_val
V["score"] = base["val_score"]
missed = V[(V["isFraud"] == 1) & (V["score"] < base["thr"])]
display(Markdown(f"**Frauds the baseline misses on validation** (flag threshold {base['thr']:.3f}):"))
display(missed[["type", "amount", "oldbalanceOrg", "newbalanceOrig", "errorBalanceOrig", "errorBalanceDest", "origEmptied", "score"]]
        .style.format({"amount": "{:,.0f}", "oldbalanceOrg": "{:,.0f}", "newbalanceOrig": "{:,.0f}",
                       "errorBalanceOrig": "{:,.0f}", "errorBalanceDest": "{:,.0f}", "score": "{:.3f}"}))

**Frauds the baseline misses on validation** (flag threshold 0.506):

,type,amount,oldbalanceOrg,newbalanceOrig,errorBalanceOrig,errorBalanceDest,origEmptied,score
410176,TRANSFER,"1,395,851",0,0,"-1,395,851",-0,1,0.005
551,CASH_OUT,"132,843","4,499",0,"-128,344",0,1,0.041
246825,CASH_OUT,"89,571","4,506",0,"-85,066",0,1,0.007
323331,CASH_OUT,"112,486",0,0,"-112,486",0,1,0.006
331652,CASH_OUT,"577,419",0,0,"-577,419",0,1,0.049


In [9]:
T = X.iloc[train_idx][ENG].copy()
T["isFraud"] = y_train
same_amount = T["amount"] == T["oldbalanceOrg"]
segments = {
    "amount == old origin balance (whole balance moved)": same_amount,
    "origin emptied, but amount != old balance": ~same_amount & (T["origEmptied"] == 1),
    "   ... and old origin balance == 0": ~same_amount & (T["origEmptied"] == 1) & (T["oldbalanceOrg"] == 0),
    "   ... and old origin balance > 0 (amount > old balance)": ~same_amount & (T["origEmptied"] == 1) & (T["oldbalanceOrg"] > 0),
}
seg = pd.DataFrame([{"segment (training set)": name, "rows": int(m.sum()), "frauds": int(T.loc[m, "isFraud"].sum()),
                     "fraud rate": T.loc[m, "isFraud"].mean()} for name, m in segments.items()])
display(seg.style.format({"rows": "{:,}", "frauds": "{:,}", "fraud rate": "{:.4%}"}).hide(axis="index"))
print(f"frauds in the training set: {int(T['isFraud'].sum()):,}")

segment (training set),rows,frauds,fraud rate
amount == old origin balance (whole balance moved),"5,628","5,628",100.0000%
"origin emptied, but amount != old balance","1,742,013",20,0.0011%
... and old origin balance == 0,"915,854",20,0.0022%
... and old origin balance > 0 (amount > old balance),"826,159",0,0.0000%


frauds in the training set: 5,749


**What the errors say.** All five missed frauds have an emptied origin account, like almost every fraud, but the amount is
*not* the old balance: the old balance is 0 or a few thousand while the amount is far larger. In the training set the
pattern "the whole balance moved" (`amount == oldbalanceOrg`) is **100% fraud** and covers nearly all frauds. The
misses come from the rest. There, the origin account is emptied with a balance of 0 in about 916 thousand legitimate
transactions, and only 20 frauds look like that: a fraud rate of about 0.002%.

**Hypothesis and expectation.** Trees split one column against a constant, so a ratio of two columns is something they
approximate rather than see. Three own-row columns are added: `amountToOldBalanceOrig`, `amountEqualsOldBalanceOrig`,
`destWasEmpty`. **Expectation, written down before running it: little or no gain.** The strongest signal, "the whole
balance moved", is already implied by `errorBalanceOrig == 0` together with `origEmptied`, and the misses live in a
corner where, from the table above, no own-row ratio separates frauds from legitimate rows. The experiment is cheap, so
it is run anyway: a negative result here tells us the remaining errors are not a feature-formulation problem.

### 4.2 Result

In [10]:
X1 = add_ratio_features(X)
ENG1 = ENG + RATIO_COLS
it1 = run(ENG1, PARAMS, frame=X1)
it1_row = summarise(it1)

verdict, gain, by_prauc, by_cost = decide(it1_row, base_row)
compare = pd.DataFrame({"baseline": base_row, "iteration 1": it1_row})
display(compare.style.format("{:,.5f}", subset=pd.IndexSlice[["val_PR-AUC"], :]).format("{:,.3f}", subset=pd.IndexSlice[["threshold"], :]).format("{:,.0f}", subset=pd.IndexSlice[["val_FN", "val_FP", "val_missed_amount", "val_alerts", COST_KEY, "trees", "fit_seconds"], :]))
display(Markdown(f"Validation PR-AUC change **{gain:+.5f}** (2σ = {2 * sigma:.5f}; PR-AUC branch: {by_prauc}). "
                 f"Cost @ {PRIMARY_REVIEW_COST}: {base_row[COST_KEY]:,.0f} → {it1_row[COST_KEY]:,.0f} (cost branch: {by_cost}). "
                 f"**Decision by rule 3: {verdict}.**"))

,baseline,iteration 1
val_PR-AUC,0.99718,0.99730
val_FN,5,5
val_FP,0,1
val_missed_amount,"2,308,170","2,308,170"
val_alerts,"1,227","1,228"
val_cost_c100,"2,430,870","2,430,970"
threshold,0.506,0.140
trees,117,58
fit_seconds,42,25


Validation PR-AUC change **+0.00012** (2σ = 0.00052; PR-AUC branch: False). Cost @ 100: 2,430,870 → 2,430,970 (cost branch: False). **Decision by rule 3: revert.**

In [11]:
gain_share = pd.Series(it1["model"].get_booster().get_score(importance_type="gain")).reindex(ENG1).fillna(0)
gain_share = (gain_share / gain_share.sum()).sort_values(ascending=False)
display(Markdown("**Share of split gain in the iteration-1 model** (are the new columns used at all?)"))
display(gain_share.to_frame("gain share").style.format("{:.1%}"))

**Share of split gain in the iteration-1 model** (are the new columns used at all?)

,gain share
amountEqualsOldBalanceOrig,51.7%
errorBalanceOrig,17.3%
origEmptied,10.2%
newbalanceDest,7.8%
amountToOldBalanceOrig,4.2%
oldbalanceOrg,2.1%
newbalanceOrig,2.0%
hour,1.4%
destWasEmpty,1.2%
isTransfer,0.8%


In [12]:
log_iteration({
    "iteration": 1, "name": "ratio and flag features",
    "hypothesis": "trees cannot form ratios of two columns; maybe the misses need them (expected: little or no gain)",
    "change": "add amountToOldBalanceOrig, amountEqualsOldBalanceOrig, destWasEmpty (own-row fields only)",
    **it1_row, "sigma_seed": sigma, "decision": verdict,
})
CURRENT = {"name": "iteration 1" if verdict == "keep" else "baseline", "cols": ENG1 if verdict == "keep" else ENG,
           "frame": X1 if verdict == "keep" else X, "row": it1_row if verdict == "keep" else base_row, "run": it1 if verdict == "keep" else base,
           "policy": "max-F1 threshold", "params": PARAMS}
print(f"Model carried into iteration 2: {CURRENT['name']}")
pd.read_csv(ROOT / "reports" / "m2_2" / "experiment_log.csv")

Model carried into iteration 2: baseline


,iteration,name,hypothesis,change,val_PR-AUC,val_FN,val_FP,trees,fit_seconds,sigma_seed,decision,val_missed_amount,val_alerts,val_cost_c100,threshold
0,0,baseline (M2.1),the refactored package reproduces the J1 model,"none: J1 tuned XGBoost, engineered features, m...",0.9972,5,0,117,42.5000,0.0003,reference,"2,308,170.0900","1,227.0000","2,430,870.0900",0.5057
1,1,ratio and flag features,trees cannot form ratios of two columns; maybe...,"add amountToOldBalanceOrig, amountEqualsOldBal...",0.9973,5,1,58,25.1000,0.0003,revert,"2,308,170.0900","1,228.0000","2,430,970.0900",0.1404
2,2,cost-based threshold,missed frauds carry most of the cost and score...,choose the threshold on validation by minimum ...,0.9972,3,589,117,18.9000,0.0003,keep,"1,597,908.4700","1,818.0000","1,779,708.4700",0.0406
3,3,"second-stage search, full training set",J1 searched 20 configurations on 30% of the tr...,"12-config grid around the J1 winner, scored by...",0.9971,3,1145,78,33.8000,0.0003,revert,"1,597,908.4700","2,374.0000","1,835,308.4700",0.0300


### 4.3 Reading the result
- **No accuracy gain, as expected.** Validation PR-AUC moved by +0.00012, less than one σ (0.00026) and far below the
  2σ bar. The same 5 frauds are missed (the same 2.31M of fraud money), and there is one more false alarm, so expected
  cost rose by 100. Rule 3 says **revert**.
- **The model used the new columns, but only to say the same thing more cheaply.** `amountEqualsOldBalanceOrig` takes
  about half of the split gain: it is a one-column version of the "whole balance moved" test the baseline built from
  `errorBalanceOrig == 0` and `origEmptied`. The result is a model half the size (58 trees instead of 117, 13 s instead
  of 24 s) with the same errors. Rule 3 does not score speed, so the rule is applied as written; the saving is noted for
  the report as a possible later change, not claimed as an improvement.
- **What this rules out.** The remaining errors are not a problem of how the own-row columns are formulated. They sit in
  the corner of §4.1 where legitimate and fraudulent rows look the same row by row. Telling them apart would need
  information from outside the row (for example an account's earlier transactions), which this work does not use.

## 5. Iteration 2 — a cost-based threshold instead of max F1

### 5.1 Hypothesis
The baseline flags a transaction when its score is at least the **max-F1 threshold** (0.506). The frauds it misses
score only 0.005–0.049 (§4.1), far below that, and they carry 95% of the baseline's cost at 100 units per alert (§3).
F1 weighs a missed fraud like a false alarm; the cost metric does not. **Hypothesis:** choosing the threshold on the
validation set to minimise expected cost, instead of maximising F1, lowers the cost. The scores, and therefore PR-AUC,
do not change; only the operating point does.

**A tightening of rule 3, written before this result was computed.** A threshold chosen on the validation rows is also
judged on those rows, and the cost-optimal cut can never cost more there than the F1 cut (the F1 cut is one of the
candidates). So "the cost fell" proves nothing by itself. For a threshold change the cost branch of rule 3 is therefore
judged **cross-fitted**: the validation set is split into two stratified halves, and each half is judged with a
threshold chosen on the *other* half. The change is kept only if the cross-fitted cost is lower too.

### 5.2 Thresholds and costs at three review costs

In [13]:
scores, f1_thr = CURRENT["run"]["val_score"], CURRENT["run"]["thr"]
rows = []
for c in REVIEW_COSTS:
    thr_c = best_cost_threshold(y_val, amount_val, scores, c)
    for label, thr in (("max-F1 threshold", f1_thr), ("min-cost threshold", thr_c)):
        r = cost_table(scores, thr)
        rows.append({"review cost": c, "operating point": label, "threshold": thr, "alerts": r["alerts"],
                     "missed frauds": r["missed frauds"], "missed amount": r["missed amount"], "expected cost": r[f"cost @ {c}"]})
display(pd.DataFrame(rows).style.format({"threshold": "{:.4f}", "alerts": "{:,}", "missed frauds": "{:,}",
                                         "missed amount": "{:,.0f}", "expected cost": "{:,.0f}"}).hide(axis="index"))

review cost,operating point,threshold,alerts,missed frauds,missed amount,expected cost
10,max-F1 threshold,0.5057,"1,227",5,"2,308,170","2,320,440"
10,min-cost threshold,0.0406,"1,818",3,"1,597,908","1,616,088"
100,max-F1 threshold,0.5057,"1,227",5,"2,308,170","2,430,870"
100,min-cost threshold,0.0406,"1,818",3,"1,597,908","1,779,708"
1000,max-F1 threshold,0.5057,"1,227",5,"2,308,170","3,535,170"
1000,min-cost threshold,0.0494,"1,487",4,"1,730,751","3,217,751"


### 5.3 Cross-fitted check (each half judged with a threshold chosen on the other half)

In [14]:
def crossfit_costs(scores, c, seed=SEED):
    """Expected cost over all validation rows when every row is judged with a threshold chosen on the other half."""
    total = {"max-F1 threshold": 0.0, "min-cost threshold": 0.0}
    for fit_part, judge_part in StratifiedKFold(2, shuffle=True, random_state=seed).split(scores, y_val):
        thresholds = {"max-F1 threshold": best_f1_threshold(y_val[fit_part], scores[fit_part]),
                      "min-cost threshold": best_cost_threshold(y_val[fit_part], amount_val[fit_part], scores[fit_part], c)}
        for label, thr in thresholds.items():
            total[label] += expected_cost(y_val[judge_part], amount_val[judge_part], scores[judge_part] >= thr, c)
    return total


cf = pd.DataFrame({c: crossfit_costs(scores, c) for c in REVIEW_COSTS}).T
cf.index.name = "review cost"
cf["saving"] = cf["max-F1 threshold"] - cf["min-cost threshold"]
display(cf.style.format("{:,.0f}"))

,max-F1 threshold,min-cost threshold,saving
review cost,,,
10,"2,731,055","2,323,270","407,785"
100,"2,841,215","2,459,170","382,045"
1000,"3,942,815","3,674,170","268,645"


In [16]:
thr_cost = best_cost_threshold(y_val, amount_val, scores, PRIMARY_REVIEW_COST)
it2_row = summarise(CURRENT["run"], thr=thr_cost)

verdict2, gain2, by_prauc2, by_cost_in_sample = decide(it2_row, CURRENT["row"])
by_cost_crossfit = cf.loc[PRIMARY_REVIEW_COST, "min-cost threshold"] < cf.loc[PRIMARY_REVIEW_COST, "max-F1 threshold"]
verdict2 = "keep" if by_cost_in_sample and by_cost_crossfit else "revert"

compare2 = pd.DataFrame({"before (max-F1 threshold)": CURRENT["row"], "iteration 2 (min-cost threshold)": it2_row})
display(compare2.style.format("{:,.3f}", subset=pd.IndexSlice[["threshold"], :]).format("{:,.5f}", subset=pd.IndexSlice[["val_PR-AUC"], :]).format("{:,.0f}", subset=pd.IndexSlice[["val_FN", "val_FP", "val_missed_amount", "val_alerts", COST_KEY, "trees", "fit_seconds"], :]))
display(Markdown(f"Cost @ {PRIMARY_REVIEW_COST}: {CURRENT['row'][COST_KEY]:,.0f} → {it2_row[COST_KEY]:,.0f} on the full validation set (falls: {by_cost_in_sample}); "
                 f"cross-fitted {cf.loc[PRIMARY_REVIEW_COST, 'max-F1 threshold']:,.0f} → {cf.loc[PRIMARY_REVIEW_COST, 'min-cost threshold']:,.0f} (falls: {by_cost_crossfit}). "
                 f"PR-AUC change {gain2:+.5f} (the scores are the same). **Decision by rule 3 as tightened: {verdict2}.**"))

,before (max-F1 threshold),iteration 2 (min-cost threshold)
val_PR-AUC,0.99718,0.99718
val_FN,5,3
val_FP,0,589
val_missed_amount,"2,308,170","1,597,908"
val_alerts,"1,227","1,818"
val_cost_c100,"2,430,870","1,779,708"
threshold,0.506,0.041
trees,117,117
fit_seconds,42,42


Cost @ 100: 2,430,870 → 1,779,708 on the full validation set (falls: True); cross-fitted 2,841,215 → 2,459,170 (falls: True). PR-AUC change +0.00000 (the scores are the same). **Decision by rule 3 as tightened: keep.**

In [17]:
log_iteration({
    "iteration": 2, "name": "cost-based threshold",
    "hypothesis": "missed frauds carry most of the cost and score far below the max-F1 threshold; a lower, cost-chosen threshold should cost less",
    "change": f"choose the threshold on validation by minimum expected cost at {PRIMARY_REVIEW_COST} units per alert instead of max F1; same scores",
    **it2_row, "sigma_seed": sigma, "decision": verdict2,
})
if verdict2 == "keep":
    CURRENT.update({"name": CURRENT["name"] + " + cost-based threshold", "row": it2_row, "policy": "min-cost threshold", "thr": thr_cost})
else:
    CURRENT["thr"] = CURRENT["run"]["thr"]
print(f"Model and operating point carried into iteration 3: {CURRENT['name']} ({CURRENT['policy']}, threshold {CURRENT['thr']:.4f})")
pd.read_csv(ROOT / "reports" / "m2_2" / "experiment_log.csv")

Model and operating point carried into iteration 3: baseline + cost-based threshold (min-cost threshold, threshold 0.0406)


,iteration,name,hypothesis,change,val_PR-AUC,val_FN,val_FP,trees,fit_seconds,sigma_seed,decision,val_missed_amount,val_alerts,val_cost_c100,threshold
0,0,baseline (M2.1),the refactored package reproduces the J1 model,"none: J1 tuned XGBoost, engineered features, m...",0.9972,5,0,117,42.5000,0.0003,reference,"2,308,170.0900","1,227.0000","2,430,870.0900",0.5057
1,1,ratio and flag features,trees cannot form ratios of two columns; maybe...,"add amountToOldBalanceOrig, amountEqualsOldBal...",0.9973,5,1,58,25.1000,0.0003,revert,"2,308,170.0900","1,228.0000","2,430,970.0900",0.1404
2,2,cost-based threshold,missed frauds carry most of the cost and score...,choose the threshold on validation by minimum ...,0.9972,3,589,117,42.5000,0.0003,keep,"1,597,908.4700","1,818.0000","1,779,708.4700",0.0406
3,3,"second-stage search, full training set",J1 searched 20 configurations on 30% of the tr...,"12-config grid around the J1 winner, scored by...",0.9971,3,1145,78,33.8000,0.0003,revert,"1,597,908.4700","2,374.0000","1,835,308.4700",0.0300


### 5.4 Reading the result
- **What changed.** The threshold falls from 0.506 to 0.041. The same model now catches 2 more frauds (3 missed instead
  of 5, 0.71M less missed money) at the price of 589 false alarms: 1,818 alerts instead of 1,227, so precision on the
  validation set falls from 100% to about 68%. PR-AUC is unchanged because the scores are the same.
- **How much of the saving is real.** At 100 units per alert, expected cost falls 27% on the validation rows the
  threshold was chosen on (2.43M → 1.78M) but 13% cross-fitted (2.84M → 2.46M). About half of the in-sample gain is
  optimism: the chosen cut (0.0406) sits exactly at the score of one missed fraud, and with only 5 misses on the
  validation set the threshold is decided by two transactions. The cross-fitted saving is positive at all three review
  costs (about 0.41M, 0.38M and 0.27M), so the direction is not an artefact of the assumed cost, but its size is.
- **What it depends on.** The whole gain rests on the review cost being small next to the size of a missed fraud. If an
  alert is expensive (1,000 units) the gain shrinks; if fraud amounts were small it could vanish. The review cost is an
  assumption (§3), and the choice of operating point is a business decision, not a modelling one.
- **Not yet verified.** Nothing here has touched the test set. Whether a threshold learned from 5 misses holds up is
  what the single test evaluation at the end checks. *(Added after that evaluation: it did not hold up. On the test set
  the cost-based threshold caught no extra fraud and cost more than the baseline; see
  [05_final_comparison.ipynb](05_final_comparison.ipynb).)*

## 6. Iteration 3 — a second-stage search on the full training set

### 6.1 Hypothesis
J1 tuned the hyper-parameters once, with 20 random configurations scored on a **30% sample** of the training set
(notebook 01, §5.2 and §7.3). A second, narrower search around the winner, using **all** of the training data, may find a
better configuration. **Expectation, written down before running it: little or no gain.** J1 found the top five
configurations within 0.0005 PR-AUC of each other, less than the fold-to-fold spread.

**Design.** 12 configurations: `max_depth` {3, 4, 5} × `learning_rate` {0.03, 0.05} × `min_child_weight` {5, 10};
`subsample` 0.8, `colsample_bytree` 0.6 and `scale_pos_weight` 18.3 stay at the J1 values. The baseline configuration
(4, 0.03, 5) is one of the 12. Each is scored by **3-fold cross-validation on the training set only**, so the validation
set plays no part in choosing the winner and the winner's curse (picking the luckiest of 12 on the same rows that judge
it) is avoided. The CV winner is then refitted once on the whole training set and judged on validation by rule 3, with the
operating point re-chosen on validation by the policy kept in iteration 2 and the cost judged cross-fitted.

### 6.2 The search

In [18]:
GRID = [{**PARAMS, "max_depth": depth, "learning_rate": lr, "min_child_weight": mcw}
        for depth, lr, mcw in product([3, 4, 5], [0.03, 0.05], [5, 10])]
X_tr_cur = CURRENT["frame"].iloc[train_idx][CURRENT["cols"]]
folds = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_tr_cur, y_train))

t0 = time.time()
cv_rows = []
for params in GRID:
    fold_scores, fold_trees = [], []
    for fit_i, hold_i in folds:
        model, _ = fit_xgb(X_tr_cur.iloc[fit_i], y_train[fit_i], X_tr_cur.iloc[hold_i], y_train[hold_i], params)
        fold_scores.append(average_precision_score(y_train[hold_i], model.predict_proba(X_tr_cur.iloc[hold_i])[:, 1]))
        fold_trees.append(model.best_iteration + 1)
    cv_rows.append({"max_depth": params["max_depth"], "learning_rate": params["learning_rate"],
                    "min_child_weight": params["min_child_weight"], "cv PR-AUC": np.mean(fold_scores),
                    "cv std": np.std(fold_scores, ddof=1), "mean trees": np.mean(fold_trees),
                    "is baseline": params == PARAMS})
cv = pd.DataFrame(cv_rows).sort_values("cv PR-AUC", ascending=False).reset_index(drop=True)
print(f"{len(GRID)} configurations x 3 folds in {time.time() - t0:.0f}s")
display(cv.style.format({"cv PR-AUC": "{:.5f}", "cv std": "{:.5f}", "mean trees": "{:.0f}"}).hide(axis="index"))

12 configurations x 3 folds in 1342s


max_depth,learning_rate,min_child_weight,cv PR-AUC,cv std,mean trees,is baseline
4,0.050000,10,0.99812,0.00052,147,False
5,0.050000,10,0.99797,0.00074,152,False
5,0.050000,5,0.99781,0.00084,157,False
4,0.050000,5,0.99758,0.00117,93,False
5,0.030000,10,0.99748,0.00115,156,False
5,0.030000,5,0.99738,0.00115,173,False
3,0.050000,5,0.99689,0.00087,74,False
3,0.050000,10,0.99686,0.00083,53,False
4,0.030000,10,0.99685,0.00078,97,False
4,0.030000,5,0.99681,0.00069,107,True


In [19]:
best_cfg = cv.iloc[0]
cv_baseline = cv.loc[cv["is baseline"]].iloc[0]
best_params = {**PARAMS, "max_depth": int(best_cfg["max_depth"]), "learning_rate": float(best_cfg["learning_rate"]),
               "min_child_weight": int(best_cfg["min_child_weight"])}
display(Markdown(f"CV winner: depth {best_params['max_depth']}, learning rate {best_params['learning_rate']}, min_child_weight "
                 f"{best_params['min_child_weight']}: CV PR-AUC {best_cfg['cv PR-AUC']:.5f} against {cv_baseline['cv PR-AUC']:.5f} for the "
                 f"baseline configuration (fold-to-fold std about {cv_baseline['cv std']:.5f}). "
                 f"{'**The baseline configuration is itself the CV winner, so there is nothing to refit.**' if best_params == PARAMS else ''}"))

CV winner: depth 4, learning rate 0.05, min_child_weight 10: CV PR-AUC 0.99812 against 0.99681 for the baseline configuration (fold-to-fold std about 0.00069). 

### 6.3 Refit the winner and judge it on validation

In [20]:
it3 = run(CURRENT["cols"], best_params, frame=CURRENT["frame"])
if CURRENT["policy"] == "min-cost threshold":
    thr3 = best_cost_threshold(y_val, amount_val, it3["val_score"], PRIMARY_REVIEW_COST)
else:
    thr3 = it3["thr"]
it3_row = summarise(it3, thr=thr3)

verdict3, gain3, by_prauc3, by_cost_in_sample3 = decide(it3_row, CURRENT["row"])
cf_now = crossfit_costs(CURRENT["run"]["val_score"], PRIMARY_REVIEW_COST)[CURRENT["policy"]]
cf_new = crossfit_costs(it3["val_score"], PRIMARY_REVIEW_COST)[CURRENT["policy"]]
by_cost_crossfit3 = cf_new < cf_now
verdict3 = "keep" if by_prauc3 or (by_cost_in_sample3 and by_cost_crossfit3 and gain3 > -sigma) else "revert"

compare3 = pd.DataFrame({"before": CURRENT["row"], "iteration 3": it3_row})
display(compare3.style.format("{:,.3f}", subset=pd.IndexSlice[["threshold"], :]).format("{:,.5f}", subset=pd.IndexSlice[["val_PR-AUC"], :]).format("{:,.0f}", subset=pd.IndexSlice[["val_FN", "val_FP", "val_missed_amount", "val_alerts", COST_KEY, "trees", "fit_seconds"], :]))
display(Markdown(f"Validation PR-AUC change **{gain3:+.5f}** (2σ = {2 * sigma:.5f}; PR-AUC branch: {by_prauc3}). "
                 f"Cost @ {PRIMARY_REVIEW_COST}: {CURRENT['row'][COST_KEY]:,.0f} → {it3_row[COST_KEY]:,.0f} (in-sample falls: {by_cost_in_sample3}); "
                 f"cross-fitted {cf_now:,.0f} → {cf_new:,.0f} (falls: {by_cost_crossfit3}). **Decision by rule 3: {verdict3}.**"))

,before,iteration 3
val_PR-AUC,0.99718,0.99712
val_FN,3,3
val_FP,589,"1,145"
val_missed_amount,"1,597,908","1,597,908"
val_alerts,"1,818","2,374"
val_cost_c100,"1,779,708","1,835,308"
threshold,0.041,0.030
trees,117,78
fit_seconds,42,18


Validation PR-AUC change **-0.00006** (2σ = 0.00052; PR-AUC branch: False). Cost @ 100: 1,779,708 → 1,835,308 (in-sample falls: False); cross-fitted 2,459,170 → 2,486,470 (falls: False). **Decision by rule 3: revert.**

In [21]:
log_iteration({
    "iteration": 3, "name": "second-stage search, full training set",
    "hypothesis": "J1 searched 20 configurations on 30% of the training data; a narrower search on all of it might find a better one (expected: little or no gain)",
    "change": f"12-config grid around the J1 winner, scored by 3-fold CV on train; winner depth {best_params['max_depth']}, lr {best_params['learning_rate']}, min_child_weight {best_params['min_child_weight']}",
    **it3_row, "sigma_seed": sigma, "decision": verdict3,
})
if verdict3 == "keep":
    CURRENT.update({"name": CURRENT["name"] + " + second-stage search", "run": it3, "row": it3_row, "thr": thr3, "params": best_params})
print(f"Final candidate: {CURRENT['name']} ({CURRENT['policy']}, threshold {CURRENT['thr']:.4f}), params {CURRENT['params']}")
pd.read_csv(ROOT / "reports" / "m2_2" / "experiment_log.csv")

Final candidate: baseline + cost-based threshold (min-cost threshold, threshold 0.0406), params {'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.6, 'min_child_weight': 5, 'scale_pos_weight': 18.3}


,iteration,name,hypothesis,change,val_PR-AUC,val_FN,val_FP,trees,fit_seconds,sigma_seed,decision,val_missed_amount,val_alerts,val_cost_c100,threshold
0,0,baseline (M2.1),the refactored package reproduces the J1 model,"none: J1 tuned XGBoost, engineered features, m...",0.9972,5,0,117,42.5000,0.0003,reference,"2,308,170.0900","1,227.0000","2,430,870.0900",0.5057
1,1,ratio and flag features,trees cannot form ratios of two columns; maybe...,"add amountToOldBalanceOrig, amountEqualsOldBal...",0.9973,5,1,58,25.1000,0.0003,revert,"2,308,170.0900","1,228.0000","2,430,970.0900",0.1404
2,2,cost-based threshold,missed frauds carry most of the cost and score...,choose the threshold on validation by minimum ...,0.9972,3,589,117,42.5000,0.0003,keep,"1,597,908.4700","1,818.0000","1,779,708.4700",0.0406
3,3,"second-stage search, full training set",J1 searched 20 configurations on 30% of the tr...,"12-config grid around the J1 winner, scored by...",0.9971,3,1145,78,18.0000,0.0003,revert,"1,597,908.4700","2,374.0000","1,835,308.4700",0.0300


### 6.4 Reading the result
- **The search found a better configuration on the training data.** Under 3-fold CV the winner (depth 4, learning rate
  0.05, min_child_weight 10) scores 0.99812 against 0.99681 for the baseline configuration, which ranks 10th of the 12.
  That gap (+0.0013) is about two fold-to-fold standard deviations of the baseline configuration (0.0007).
- **It did not carry over to validation.** Refitted on the whole training set, the winner's validation PR-AUC is
  0.99712 against 0.99718 (−0.00006, inside the noise floor), it misses the same 3 frauds at its cost-chosen threshold,
  and it needs more false alarms to do so (1,145 against 589), so the cost is higher both in-sample and cross-fitted.
  Rule 3 says **revert**.
- **Consistent with J1.** J1 found the top configurations within the noise of each other; a wider search on more data
  gives the same picture. Hyper-parameters are not where the remaining error is. The search cost about 22 minutes of
  compute (1,342 s in the saved run, 1,471 s in an earlier one) for no gain, which is the practical lesson: no more budget on tuning this model.
- **The model carried to the final test comparison** is therefore the baseline model with the cost-based threshold from
  iteration 2.

## 7. What carries over

By rules 1–3 the surviving change is **iteration 2** (baseline model, cost-based threshold); iterations 1 and 3 were
reverted. The single test-set evaluation that rules 2 and 4 call for, with the paired bootstrap, is in
[05_final_comparison.ipynb](05_final_comparison.ipynb). The raw-feature convergence problem found along the way is in
[04_raw_feature_convergence.ipynb](04_raw_feature_convergence.ipynb).